# Convert tiff to OME-zarr method 2

In [16]:
import math

import zarr
from bioio_ome_zarr.writers import OMEZarrWriter
from bioio_ome_zarr.writers import edit_metadata

filepath_seg = "data/box1-2_all.tif"
filepath_raw = "data/mnemiopsis-box1-2.tif"

raw = tifffile.imread("data/mnemiopsis-box1-2.tif", aszarr=True)
rawzarr = zarr.open(raw, mode="r")

In [17]:
rawzarr.shape
# new axis is added? get rid of it
rawzarr = rawzarr[:, :, :, 0]
rawzarr.shape

(128, 1000, 1000)

In [23]:
# Shapes of each resolution level you want to make
level_shapes = [
    rawzarr.shape,
    [rawzarr.shape[0], rawzarr.shape[1] // 2, rawzarr.shape[2] // 2],
    [rawzarr.shape[0], rawzarr.shape[1] // 4, rawzarr.shape[2] // 4],
    [rawzarr.shape[0], rawzarr.shape[1] // 8, rawzarr.shape[2] // 8],
]

print(level_shapes)


[(128, 1000, 1000), [128, 500, 500], [128, 250, 250], [128, 125, 125]]


In [25]:
%%time

# Writer with any additional metadata required
writer = OMEZarrWriter(
    store="data/raw.ome.zarr",
    level_shapes=level_shapes,
    dtype=rawzarr.dtype,
    zarr_format=3,
    axes_names=["z", "x", "y"],
    axes_types=["space", "space", "space"],
    axes_units=["um", "um", "um"],
)

writer.write_full_volume(rawzarr)

CPU times: user 827 ms, sys: 80.6 ms, total: 908 ms
Wall time: 498 ms


## add metadata

In [26]:
writer.write_full_volume(rawzarr)

root = zarr.open_group("data/raw.ome.zarr", mode="a")

root.attrs["sample"] = {
    "species": "Mnemiopsis leidyi",
    "body part": "aboral organ",
    "preparation": "HPF, array tomography"
}
